# Survival Table Construction
Build a patient-level survival table (one row per case: `event`, `time`, and the list of slide files per patient) from the TCGA-LUAD GDC data.

**Cohort:** TCGA-LUAD diagnostic slides  
**Data source:** [GDC REST API](https://api.gdc.cancer.gov) — no local TSV files needed  
**Input:** `manifests/gdc_manifest_full_luad_dx.txt` (541 files) + slides on disk  
**Result:** 468 cases, 540 files (1 SVS not downloaded; 9 cases dropped for missing follow-up time)

## Imports

In [23]:
%matplotlib inline
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

plt.rcdefaults()
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({"figure.dpi": 150})

## Paths
- **MANIFEST_PATH** — `gdc_manifest_full_luad_dx.txt` — 541 file IDs downloaded from GDC Portal
- **SLIDES_DIR** — `data/slides/` — downloaded SVS files (one sub-folder per file ID)
- **OUT_PATH** — `data/interim/matched_clinical_slides.csv` — output survival table

GDC API endpoints used:
- `/files` — maps each file ID to its case ID and filename
- `/cases` — returns clinical data (vital_status, days_to_death, days_to_last_follow_up) per case

In [24]:
ROOT          = Path("..")
MANIFEST_PATH = ROOT / "manifests" / "gdc_manifest_full_luad_dx.txt"
SLIDES_DIR    = ROOT / "data" / "slides"
OUT_PATH      = ROOT / "data" / "interim" / "matched_clinical_slides.csv"

GDC_FILES_URL = "https://api.gdc.cancer.gov/files"
GDC_CASES_URL = "https://api.gdc.cancer.gov/cases"
BATCH         = 100  # max items per API request

## 1. Check Slides on Disk
Read all file IDs from the manifest and keep only those that have an actual `.svs` file in `data/slides/`.

In [25]:
manifest = pd.read_csv(MANIFEST_PATH, sep="\t")
all_file_ids = manifest["id"].tolist()
print(f"Manifest file count: {len(all_file_ids)}")

valid_file_ids = [
    fid for fid in all_file_ids
    if list((SLIDES_DIR / fid).glob("*.svs"))
]
print(f"Files with SVS on disk: {len(valid_file_ids)} / {len(all_file_ids)}")

Manifest file count: 541
Files with SVS on disk: 540 / 541


## 2. Fetch File → Case Mapping (GDC API)
Query GDC `/files` endpoint in batches of 100 to get `file_id → case_id` and `file_name` for every SVS on disk.

In [26]:
def gdc_post(url, payload, retries=3):
    for attempt in range(retries):
        try:
            r = requests.post(url, json=payload, timeout=60)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"  Retry {attempt+1}/{retries}: {e}")
            time.sleep(5)

file_case_rows = []
for i in range(0, len(valid_file_ids), BATCH):
    batch = valid_file_ids[i : i + BATCH]
    data = gdc_post(GDC_FILES_URL, {
        "filters": {"op": "in", "content": {"field": "file_id", "value": batch}},
        "fields": "file_id,file_name,cases.case_id,cases.submitter_id",
        "size": BATCH,
    })
    for hit in data["data"]["hits"]:
        for case in hit.get("cases", []):
            file_case_rows.append({
                "file_id":      hit["file_id"],
                "file_name":    hit["file_name"],
                "case_id":      case["case_id"],
                "submitter_id": case["submitter_id"],
            })
    print(f"  {min(i+BATCH, len(valid_file_ids))}/{len(valid_file_ids)} done")

file_case_df = pd.DataFrame(file_case_rows)
print(f"\nFile-case rows: {len(file_case_df)}")
print(f"Unique cases:   {file_case_df['case_id'].nunique()}")
file_case_df.head()

  100/540 done
  200/540 done
  300/540 done
  400/540 done
  500/540 done
  540/540 done

File-case rows: 540
Unique cases:   477


,file_id,file_name,case_id,submitter_id
0,6a0ea716-a5f2-47f3-880b-537a5cdc2324,TCGA-86-8074-01Z-00-DX1.0c34b434-8701-4060-a4e...,482eb2a7-6fed-4e4c-b1b6-14d6d869f855,TCGA-86-8074
1,c5cc9280-d4c9-4090-9fb1-1d1cc5e2f8d6,TCGA-78-7535-01Z-00-DX1.c4ca06f3-22d1-4e39-85c...,46592b7b-6968-42a6-83af-0917c9f4a9a5,TCGA-78-7535
2,1ec9435b-6056-4d34-802d-a4d20208f60e,TCGA-86-8358-01Z-00-DX1.C50100F8-9414-4A06-BED...,467d96d7-4a42-4116-8c32-0e4b34f96d8a,TCGA-86-8358
3,19b89589-cbb4-4b6a-bd21-257c04c531fe,TCGA-78-7158-01Z-00-DX1.74db3dcd-2715-4ce7-8e4...,501c987e-d1eb-48a9-89eb-72a5062c90b4,TCGA-78-7158
4,7bc387ff-930d-49d4-b93c-b1cc61d3ca65,TCGA-86-A4D0-01Z-00-DX1.165461AD-A8DA-4B7B-87C...,528a0fce-0719-4b19-a69d-3f18681696c5,TCGA-86-A4D0


## 3. Fetch Clinical Data (GDC API)
For each unique case, fetch `vital_status`, `days_to_death`, and `days_to_last_follow_up`.  
Some patients have multiple diagnoses rows — we take the `max` of `days_to_last_follow_up` across them.

In [27]:
case_ids = file_case_df["case_id"].unique().tolist()
print(f"Unique cases: {len(case_ids)}")

clinical_rows = []
for i in range(0, len(case_ids), BATCH):
    batch = case_ids[i : i + BATCH]
    data = gdc_post(GDC_CASES_URL, {
        "filters": {"op": "in", "content": {"field": "case_id", "value": batch}},
        "fields": (
            "case_id,submitter_id,"
            "demographic.vital_status,"
            "demographic.days_to_death,"
            "diagnoses.days_to_last_follow_up"
        ),
        "size": BATCH,
        "expand": "demographic,diagnoses",
    })
    for hit in data["data"]["hits"]:
        demo  = hit.get("demographic") or {}
        diags = hit.get("diagnoses") or []
        lfu_vals = [d.get("days_to_last_follow_up") for d in diags
                    if d.get("days_to_last_follow_up") is not None]
        clinical_rows.append({
            "case_id":                hit["case_id"],
            "vital_status":           demo.get("vital_status"),
            "days_to_death":          demo.get("days_to_death"),
            "days_to_last_follow_up": max(lfu_vals) if lfu_vals else None,
        })
    print(f"  {min(i+BATCH, len(case_ids))}/{len(case_ids)} done")

clinical_df = pd.DataFrame(clinical_rows)
print(f"\nClinical data rows: {len(clinical_df)}")
clinical_df.head()

Unique cases: 477


  100/477 done
  200/477 done
  300/477 done
  400/477 done
  477/477 done

Clinical data rows: 477


,case_id,vital_status,days_to_death,days_to_last_follow_up
0,e51f717f-72e8-400e-9f35-fc5dc32fdf71,Alive,NaN,2595.0
1,c650b1ff-8a4c-4ee9-b7c1-268c28c83827,Dead,503.0,503.0
2,8609edfc-119d-4d63-9188-c86aabd5ca52,Alive,NaN,993.0
3,1b354837-4925-4480-ac32-6b44d0957314,Alive,NaN,131.0
4,199386c2-bb53-4fad-a1b6-59ab216a4a50,Alive,NaN,2065.0


# Create Survival Table

- We merge the deduplicated clinical data to the sample sheet (which indexes slides to cases), then group by case to bag all slides per patient into a list.

### Set Event (Dead or Alive) to Binary
- Cox regression requires numbers: `event = 1` (Dead), `event = 0` (Alive)

In [28]:
clinical_df["days_to_death"]          = pd.to_numeric(clinical_df["days_to_death"],          errors="coerce")
clinical_df["days_to_last_follow_up"] = pd.to_numeric(clinical_df["days_to_last_follow_up"], errors="coerce")

clinical_df["event"] = (clinical_df["vital_status"] == "Dead").astype(int)

print(f"Dead   (event=1): {clinical_df['event'].sum()}")
print(f"Alive  (event=0): {(clinical_df['event']==0).sum()}")

Dead   (event=1): 168
Alive  (event=0): 309


### Set Event (Dead or Alive) to Binary
- Cox regression requires numbers so we set event: 1="Dead" and 0="Alive"

In [29]:
clinical_df["time"] = clinical_df["days_to_death"].where(
    clinical_df["event"] == 1,
    clinical_df["days_to_last_follow_up"],
)
clinical_df["time"] = pd.to_numeric(clinical_df["time"], errors="coerce")

print(f"Missing time: {clinical_df['time'].isna().sum()} / {len(clinical_df)}")

Missing time: 9 / 477


### Merge File Data & Clinical Data
Join `file_case_df` (file_id → case_id) with `clinical_df` (case_id → survival info).

In [30]:
merged = file_case_df.merge(clinical_df, on="case_id", how="left")
print(f"Merged rows: {len(merged)}")
merged.head()

Merged rows: 540


,file_id,file_name,case_id,submitter_id,vital_status,days_to_death,days_to_last_follow_up,event,time
0,6a0ea716-a5f2-47f3-880b-537a5cdc2324,TCGA-86-8074-01Z-00-DX1.0c34b434-8701-4060-a4e...,482eb2a7-6fed-4e4c-b1b6-14d6d869f855,TCGA-86-8074,Alive,NaN,24.0,0,24.0
1,c5cc9280-d4c9-4090-9fb1-1d1cc5e2f8d6,TCGA-78-7535-01Z-00-DX1.c4ca06f3-22d1-4e39-85c...,46592b7b-6968-42a6-83af-0917c9f4a9a5,TCGA-78-7535,Dead,949.0,949.0,1,949.0
2,1ec9435b-6056-4d34-802d-a4d20208f60e,TCGA-86-8358-01Z-00-DX1.C50100F8-9414-4A06-BED...,467d96d7-4a42-4116-8c32-0e4b34f96d8a,TCGA-86-8358,Alive,NaN,653.0,0,653.0
3,19b89589-cbb4-4b6a-bd21-257c04c531fe,TCGA-78-7158-01Z-00-DX1.74db3dcd-2715-4ce7-8e4...,501c987e-d1eb-48a9-89eb-72a5062c90b4,TCGA-78-7158,Dead,179.0,179.0,1,179.0
4,7bc387ff-930d-49d4-b93c-b1cc61d3ca65,TCGA-86-A4D0-01Z-00-DX1.165461AD-A8DA-4B7B-87C...,528a0fce-0719-4b19-a69d-3f18681696c5,TCGA-86-A4D0,Dead,116.0,116.0,1,116.0


### Group by Case (One Row per Patient)
Aggregate slides into a list per patient.

In [31]:
survival_table = (
    merged.groupby("submitter_id")
    .agg(
        case_id      =("case_id",      "first"),
        submitter_id =("submitter_id", "first"),
        file_ids     =("file_id",      list),
        file_names   =("file_name",    list),
        n_slides     =("file_id",      "count"),
        vital_status =("vital_status", "first"),
        event        =("event",        "first"),
        time         =("time",         "first"),
    )
    .reset_index(drop=True)
)

# Drop patients with no time data
survival_table = survival_table[survival_table["time"].notna()].copy()

print(f"Final survival table: {len(survival_table)} patients")
print(f"\t{int(survival_table['event'].sum())} events (deaths)")
print(f"\t{int((survival_table['event']==0).sum())} censored (alive)")
print(f"\tEvent rate: {survival_table['event'].mean()*100:.1f}%")
print(f"\nSlides per patient:")
print(survival_table['n_slides'].value_counts().sort_index())

Final survival table: 468 patients
	164 events (deaths)
	304 censored (alive)
	Event rate: 35.0%

Slides per patient:
n_slides
1     447
2       9
3       1
4       5
5       1
6       1
7       2
8       1
10      1
Name: count, dtype: int64


### Create and Save Survival Table as CSV

In [32]:
survival_table.to_csv(OUT_PATH, index=False)
print(f"Saved to: {OUT_PATH}")
survival_table.head()

Saved to: ../data/interim/matched_clinical_slides.csv


,case_id,submitter_id,file_ids,file_names,n_slides,vital_status,event,time
0,34040b83-7e8a-4264-a551-b16621843e28,TCGA-05-4244,[0b3cff7b-bbe2-40ab-aee6-cb0554940eaa],[TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-82...,1,Alive,0,0.0
1,03d09c05-49ab-4ba6-a8d7-e7ccf71fafd2,TCGA-05-4245,[97f2808a-be03-4dd0-b291-dbd797d659e1],[TCGA-05-4245-01Z-00-DX1.36ff5403-d4bb-4415-b2...,1,Alive,0,730.0
2,4addf05f-3668-4b3f-a17f-c0227329ca52,TCGA-05-4249,[b6fea18e-9615-49c1-a10b-1d466639c2bd],[TCGA-05-4249-01Z-00-DX1.9fce0297-cc19-4c04-87...,1,Alive,0,1523.0
3,f98ecd8a-b878-4f53-b911-20cd8e17281c,TCGA-05-4250,[586edaf8-4366-4ba8-a8f1-a79ed9c1f448],[TCGA-05-4250-01Z-00-DX1.90f67fdf-dff9-46ca-af...,1,Dead,1,121.0
4,3434b91a-c05f-460f-a078-7b1bb6e7085d,TCGA-05-4382,[9eb264eb-548e-4452-955a-fb5572e43cf6],[TCGA-05-4382-01Z-00-DX1.76b49a4c-dbbb-48b0-b6...,1,Alive,0,607.0


### Validate Data
- We check for missing time and slides for each patient.

In [33]:
print(f"Manifest files:              {len(all_file_ids)}")
print(f"Files with SVS on disk:      {len(valid_file_ids)}")
print(f"Cases with slides:           {file_case_df['case_id'].nunique()}")
print(f"Final table:                 {len(survival_table)} cases (slides + time available)")
print(f"\nDropped:")
print(f"\t{len(all_file_ids) - len(valid_file_ids)} files — SVS not on disk")
print(f"\t{file_case_df['case_id'].nunique() - len(survival_table)} cases — missing follow-up time")

Manifest files:              541
Files with SVS on disk:      540
Cases with slides:           477
Final table:                 468 cases (slides + time available)

Dropped:
	1 files — SVS not on disk
	9 cases — missing follow-up time
